# Product portfolio — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## Setup — caricamento e controlli

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'axes.titleweight':'semibold', 'axes.labelsize':10, 'font.size':10})
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.max_rows', 80)

candidates = [
    Path.cwd() / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent.parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd() / 'adidas_sales_features.csv',
    Path.cwd().parent / 'adidas_sales_features.csv',
]
file_input = next((p for p in candidates if p.is_file()), None)
if file_input is None:
    raise FileNotFoundError('Manca output/adidas_sales_features.csv. Esegui i notebook 01 e 02 del progetto, poi avvia questo notebook dalla root o dalla cartella notebooks/.')
df = pd.read_csv(file_input, parse_dates=['invoice_date'], dtype={'retailer_id': 'string'})
expected = {'retailer','invoice_date','region','state','city','product','price_per_unit',
            'units_sold','total_sales','operating_profit','operating_margin','sales_method',
            'year','month_number','month_name_it','year_month','weekday_number',
            'weekday_name_it','is_weekend','season_it','product_gender','product_category'}
assert expected.issubset(df.columns), f'Colonne mancanti: {expected-set(df.columns)}'
assert not df['invoice_date'].isna().any(), 'Date mancanti'
assert np.allclose(df['total_sales'], df['price_per_unit']*df['units_sold'],atol=.011), 'Ricavi non coerenti'
assert np.allclose(df['operating_profit'],df['total_sales']*df['operating_margin'],atol=.011), 'Profitto non coerente'

EURO_MONTH=['Gennaio','Febbraio','Marzo','Aprile','Maggio','Giugno','Luglio','Agosto','Settembre','Ottobre','Novembre','Dicembre']
DAYS=['Lunedì','Martedì','Mercoledì','Giovedì','Venerdì','Sabato','Domenica']
SEASONS=['Inverno','Primavera','Estate','Autunno']

def units(x): return f'{x:,.0f}'.replace(',', '.')
def usd(x): return f'${x:,.2f}' if abs(x)<1000 else f'${x:,.0f}'
def percent(x): return 'N/D' if pd.isna(x) else f'{x:,.2f}%'
def safe(a,b): return np.nan if b==0 or pd.isna(b) else a/b
def avg(x): return f'{x:,.2f}'
def note(message): display(Markdown('**Risultato:** '+str(message)))
def caution(message): display(Markdown('**Limite interpretativo:** '+str(message)))
def data_table(frame, rows=80):
    display(frame.head(rows).reset_index(drop=True).round(2))
    if len(frame)>rows:
        display(Markdown(f'_Visualizzate {rows} righe su {len(frame)}. La tabella completa rimane disponibile nella variabile Python._'))

def summary(dims, data=None):
    frame=df if data is None else data
    group=frame.groupby(dims,dropna=False,observed=True)
    result=group.agg(units=('units_sold','sum'),revenue=('total_sales','sum'),
                     profit=('operating_profit','sum'),rows=('units_sold','size'),
                     price_mean=('price_per_unit','mean'),days=('invoice_date','nunique')).reset_index()
    result['margin_pct']=np.where(result.revenue!=0,100*result.profit/result.revenue,np.nan)
    result['realized_price']=np.where(result.units!=0,result.revenue/result.units,np.nan)
    result['profit_unit']=np.where(result.units!=0,result.profit/result.units,np.nan)
    return result

def dominance(frame,dims,category,measure='units'):
    x=frame.sort_values(dims+[measure,category],ascending=[True]*len(dims)+[False,True])
    first=x.drop_duplicates(subset=dims,keep='first').copy()
    top=frame.groupby(dims)[measure].transform('max')
    tied=frame[frame[measure].eq(top)].groupby(dims,dropna=False)[category].nunique().reset_index(name='ex_aequo')
    return first.merge(tied,on=dims,how='left')

def bar(frame,label,value,title,top=12,ascending=False,ax=None):
    d=frame.sort_values(value,ascending=ascending).head(top).copy()
    if ax is None: fig,ax=plt.subplots(figsize=(11,max(4,0.40*len(d)+1.5)))
    else: fig=ax.figure
    d=d.iloc[::-1]
    ax.barh(d[label].astype(str),d[value]);ax.set_title(title);ax.set_xlabel(value.replace('_',' ').title());ax.grid(axis='x',alpha=.25)
    ax.tick_params(axis='y',labelsize=9)
    fig.tight_layout()
    return ax

def twobars(frame,label,metrics,titles,top=12):
    fig,axes=plt.subplots(1,2,figsize=(15,max(4,0.43*min(top,len(frame))+1)))
    for ax,m,t in zip(axes,metrics,titles):bar(frame,label,m,t,top=top,ax=ax)
    plt.show()

def stack(data,index,columns,measure,title,top=14,shares=True):
    g=data.pivot_table(index=index,columns=columns,values=measure,aggfunc='sum',fill_value=0)
    if g.empty:
        note('Non sono presenti dati per questo confronto.');return
    g=g.loc[g.sum(axis=1).sort_values(ascending=False).head(top).index]
    if shares: g=g.div(g.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    ax=g.plot(kind='bar',stacked=True,figsize=(max(10, min(16,len(g)*0.7+5)),5.5),width=.80)
    ax.set_title(title);ax.set_ylabel('Quota sul volume (%)' if shares else measure)
    ax.set_xlabel('');ax.legend(title=columns,bbox_to_anchor=(1.01,1),loc='upper left')
    plt.xticks(rotation=40,ha='right');plt.tight_layout();plt.show()

def heat(data,index,columns,value,title,top=25,normalize=False):
    piv=data.pivot_table(index=index,columns=columns,values=value,aggfunc='sum',fill_value=0)
    if piv.empty: return
    piv=piv.loc[piv.sum(axis=1).sort_values(ascending=False).head(top).index]
    if normalize: piv=piv.div(piv.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    fig,ax=plt.subplots(figsize=(max(8,1.15*len(piv.columns)+4),max(4,.36*len(piv)+1.8)))
    im=ax.imshow(piv.to_numpy(),aspect='auto',cmap='Blues')
    ax.set_xticks(np.arange(len(piv.columns)),piv.columns.astype(str),rotation=35,ha='right')
    ax.set_yticks(np.arange(len(piv.index)),piv.index.astype(str));ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.80,label='Quota (%)' if normalize else value)
    fig.tight_layout();plt.show()

def lineplot(frame,x,y,title,color_key=None):
    fig,ax=plt.subplots(figsize=(12,5))
    if color_key is None:
        t=frame.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3)
    else:
        for name,piece in frame.groupby(color_key,observed=True):
            t=piece.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3,label=str(name))
        ax.legend(title=color_key)
    ax.set_title(title);ax.set_ylabel(y);ax.tick_params(axis='x',rotation=70,labelsize=8)
    ax.grid(alpha=.25);fig.tight_layout();plt.show()

note(f'Caricate {units(len(df))} righe • periodo {df.invoice_date.min():%Y-%m-%d} — {df.invoice_date.max():%Y-%m-%d} • nessun CSV verrà esportato.')

## What products and categories sell the most units?

In [ ]:
p=summary(['product']).sort_values('units',ascending=False);c=summary(['product_category']).sort_values('units',ascending=False)
r=p.iloc[0];note(f"Prodotto con più unità: {r['product']} ({units(r.units)}). Macro-categoria leader: {c.iloc[0].product_category} ({units(c.iloc[0].units)}).")
twobars(p,'product',['units','revenue'],['Unità per tipologia','Fatturato per tipologia'])
data_table(p);data_table(c)

## Which products lead in revenue and profitability?

In [ ]:
p=summary(['product']);r=p.loc[p.revenue.idxmax()];s=p.loc[p.profit.idxmax()]
note(f"Ricavi: primo {r['product']} ({usd(r.revenue)}). Profitto: primo {s['product']} ({usd(s.profit)}). I margini sono rapporti ponderati ricavi.")
fig,axes=plt.subplots(1,3,figsize=(20,6));bar(p,'product','revenue','Ricavi (USD)',ax=axes[0]);bar(p,'product','profit','Profitto (USD)',ax=axes[1]);bar(p,'product','margin_pct','Margine (%)',ax=axes[2]);plt.show()
data_table(p.sort_values('profit',ascending=False))

## How does product demand vary by place and season?

In [ ]:
rg=summary(['region','product']);st=summary(['region','state','product']);ct=summary(['region','state','city','product']);season=summary(['season_it','product'])
r=rg.loc[rg.units.idxmax()]
note(f"Combinazione regione × prodotto con più unità: {r.region}, {r['product']} ({units(r.units)}). Le tabelle riportano il dettaglio territoriale e stagionale.")
fig,axes=plt.subplots(1,2,figsize=(18,6))
for ax,tab,index,title in [(axes[0],rg,'region','Regione'),(axes[1],season,'season_it','Stagione')]:
    pivot=tab.pivot(index=index,columns='product',values='units').fillna(0)
    pivot.plot(kind='bar',stacked=True,ax=ax,legend=False);ax.set_title('Volumi per '+title);ax.set_ylabel('Unità');ax.tick_params(axis='x',rotation=30)
axes[1].legend(title='Tipologia',bbox_to_anchor=(1.01,1),loc='upper left',fontsize=7)
plt.tight_layout();plt.show()
data_table(rg);data_table(st.sort_values('units',ascending=False),rows=300);data_table(ct.sort_values('units',ascending=False),rows=350);data_table(season)

## Which product target labels dominate volume?

In [ ]:
t=summary(['product_gender']).sort_values('units',ascending=False)
r=t.iloc[0];note(f"Il target prodotto {r.product_gender} pesa {percent(r.units/t.units.sum()*100)} delle unità ({units(r.units)}). Target del prodotto ≠ sesso dell’acquirente.")
twobars(t,'product_gender',['units','margin_pct'],['Unità per target prodotto','Margine (%) per target'])
data_table(t)

## How do target labels vary by place and season?

In [ ]:
locs=[['region'],['region','state'],['region','state','city'],['season_it']]
tables=[summary(l+['product_gender']) for l in locs]
winners=[dominance(t,l,'product_gender') for t,l in zip(tables,locs)]
note('Dominanza = maggior numero di unità per target del prodotto. Un eventuale ex aequo è segnalato in tabella.')
heat(tables[0],'region','product_gender','units','Target prodotto per regione (%)',normalize=True)
for l,w in zip(locs,winners):data_table(w[l+['product_gender','units','ex_aequo']])

## How do list prices compare with realized unit value?

In [ ]:
p=summary(['product']);c=summary(['product_category']);g=summary(['product_gender']);allparts=[]
for label,t in [('Prodotto',p),('Categoria',c),('Target',g)]:
    name={'Prodotto':'product','Categoria':'product_category','Target':'product_gender'}[label]
    t=t.copy();t['segmento']=label;t['nome']=t[name];allparts.append(t[['segmento','nome','price_mean','realized_price','units']])
r=pd.concat(allparts,ignore_index=True)
note(f"Media semplice complessiva: {usd(df.price_per_unit.mean())}; prezzo realizzato per unità: {usd(safe(df.total_sales.sum(),df.units_sold.sum()))}. I due indicatori non sono intercambiabili.")
t=r[r.segmento=='Prodotto'].set_index('nome')[['price_mean','realized_price']];t.plot(kind='bar',figsize=(11,5));plt.title('Prezzo per prodotto: semplice vs ponderato (USD)');plt.ylabel('USD/unità');plt.xticks(rotation=35,ha='right');plt.tight_layout();plt.show()
data_table(r)